MCL

In [3]:
import markov_clustering as mc
import networkx as nx
import numpy as np
import os

def read_weighted_interaction_file(filename, min_weight=0.0):
    """Read weighted interaction file with weight filtering"""
    G = nx.Graph()
    proteins = set()
    
    with open(filename, 'r') as f:
        header = f.readline()
        if not header.startswith("protein1"):
            f.seek(0)
            
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue
                
            protein1, protein2, weight = parts[0], parts[1], float(parts[2])
            if weight >= min_weight:
                G.add_edge(protein1, protein2, weight=weight)
                proteins.update([protein1, protein2])
    
    proteins = sorted(proteins)
    adj_matrix = nx.to_numpy_array(G, nodelist=proteins)
    return adj_matrix, proteins, G

def run_mcl_clustering(adj_matrix, inflation=2.0, expansion=2):
    """Run MCL clustering"""
    result = mc.run_mcl(adj_matrix, inflation=inflation, expansion=expansion)
    return mc.get_clusters(result)

def filter_clusters(clusters, protein_list, min_size=3):
    """Filter clusters by minimum size"""
    return [sorted([protein_list[i] for i in cluster]) 
            for cluster in clusters if len(cluster) >= min_size]

def evaluate_clusters(clusters, G):
    """Calculate cluster metrics"""
    if not clusters:
        return 0, 0
    avg_size = np.mean([len(c) for c in clusters])
    density = np.mean([nx.density(G.subgraph(c)) for c in clusters])
    return avg_size, density

def write_complexes(output_file, complexes):
    """Write complexes to output file"""
    with open(output_file, 'w') as f:
        for i, complex_proteins in enumerate(complexes, 1):
            f.write(f"{i}\t{';'.join(complex_proteins)}\n")

# Configuration - Update these paths for your 5 files
input_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt"
]

output_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_STRING_levure.txt"
]

# Common parameters
params = {
    'min_weight': 0.2,    # Minimum interaction weight threshold
    'inflation': 2.0,     # MCL inflation parameter
    'min_size': 3         # Minimum complex size
}

# Process all files
for input_file, output_file in zip(input_files, output_files):
    try:
        print(f"\nProcessing {os.path.basename(input_file)}...")
        
        # 1. Read and prepare data
        adj_matrix, proteins, G = read_weighted_interaction_file(
            input_file, 
            min_weight=params['min_weight']
        )
        
        # 2. Run MCL clustering
        clusters = run_mcl_clustering(
            adj_matrix, 
            inflation=params['inflation']
        )
        
        # 3. Filter and process results
        complexes = filter_clusters(
            clusters, 
            proteins, 
            min_size=params['min_size']
        )
        
        # 4. Evaluate and save
        avg_size, avg_density = evaluate_clusters(complexes, G)
        
        print("\nResults:")
        print(f"- Complexes detected: {len(complexes)}")
        print(f"- Avg size: {avg_size:.2f} proteins")
        print(f"- Avg density: {avg_density:.2f}")
        
        write_complexes(output_file, complexes)
        print(f"Results saved to {output_file}")
        
        # Show sample output
        print("\nSample complexes (first 5):")
        for i, c in enumerate(complexes[:5], 1):
            print(f"{i}. {len(c)} proteins: {', '.join(c[:3])}...")

    except Exception as e:
        print(f"Error processing {input_file}: {str(e)}")

print("\nAll files processed!")


Processing weighted_BIOGRID_humain.txt...

Results:
- Complexes detected: 723
- Avg size: 4.97 proteins
- Avg density: 0.70
Results saved to /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_BIOGRID_humain.txt

Sample complexes (first 5):
1. 4 proteins: A0AV96, Q14671, Q8TB72...
2. 17 proteins: A0AVF1, Q13099, Q86WT1...
3. 3 proteins: A0AVT1, O15205, Q9H832...
4. 33 proteins: A0JLT2, O43513, O60244...
5. 3 proteins: A1KXE4, Q9GZT9, Q9H6Z9...

Processing weighted_BIOGRID_levure.txt...

Results:
- Complexes detected: 374
- Avg size: 5.36 proteins
- Avg density: 0.69
Results saved to /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCL_complexes_BIOGRID_levure.txt

Sample complexes (first 5):
1. 4 proteins: D6W196, P04710, P18238...
2. 5 proteins: O13525, P27680, P53124...
3. 11 proteins: O13539, P17629, P25555...
4. 37 proteins: O13563, P21242, P21243...
5. 3 proteins: P0

MCODE

In [2]:
import networkx as nx
from tqdm import tqdm
import os
import time
import sys

def compute_core_clustering_coefficient(graph, node):
    """Version optimisée du calcul de coefficient"""
    neighbors = list(graph.neighbors(node))
    if len(neighbors) < 2:
        return 0.0
    
    neighborhood = graph.subgraph(neighbors + [node])
    max_core = nx.k_core(neighborhood)
    
    if max_core.number_of_edges() > 0:
        n = max_core.number_of_nodes()
        return max_core.number_of_edges() / (n * (n - 1) / 2)
    return 0.0

def mcode_algorithm(graph):
    """Implémentation autonome de MCODE"""
    # Pré-traitement
    graph.remove_edges_from(nx.selfloop_edges(graph))
    
    # Vertex Weighting
    print("Calcul des poids des nœuds...")
    weights = {node: compute_core_clustering_coefficient(graph, node) 
              for node in tqdm(graph.nodes(), desc="Vertex Weighting")}
    
    # Complex Prediction
    clusters = []
    unvisited = set(graph.nodes())
    pbar = tqdm(total=len(unvisited), desc="Détection des complexes")
    
    while unvisited:
        seed = max(unvisited, key=lambda x: weights.get(x, 0), default=None)
        if not seed or weights.get(seed, 0) == 0:
            break
            
        cluster = {seed}
        frontier = {seed}
        
        while frontier:
            new_frontier = set()
            for node in frontier:
                new_frontier.update(
                    n for n in graph.neighbors(node)
                    if n in unvisited and weights.get(n, 0) >= 0.6 * weights[seed]
                )
            cluster.update(new_frontier)
            frontier = new_frontier
        
        unvisited -= cluster
        pbar.update(len(cluster))
        
        if len(cluster) >= 3:
            clusters.append(cluster)
    
    pbar.close()
    return sorted(clusters, key=lambda x: -len(x))

def process_file(input_file, output_file):
    """Pipeline complet avec gestion des erreurs"""
    print(f"\nTraitement de {os.path.basename(input_file)}")
    start_time = time.time()
    
    try:
        # Lecture du fichier
        G = nx.Graph()
        with open(input_file, 'r') as f:
            next(f)  # Skip header
            for line in tqdm(f, desc="Lecture fichier"):
                parts = line.strip().split()
                if len(parts) >= 2:  # Prendre seulement les 2 premières colonnes
                    G.add_edge(parts[0], parts[1])
        
        # Détection des complexes
        complexes = mcode_algorithm(G)
        
        # Écriture des résultats
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, 'w') as f:
            for i, cluster in enumerate(complexes, 1):
                f.write(f"{i}\t{';'.join(sorted(cluster))}\n")
        
        print(f"Terminé en {time.time()-start_time:.1f}s - {len(complexes)} complexes")
    
    except Exception as e:
        print(f"Erreur: {str(e)}")

# Liste de vos fichiers (gardez vos chemins existants)
input_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt"
]

output_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCODE_complexes_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCODE_complexes_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCODE_complexes_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCODE_complexes_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/MCODE_complexes_STRING_levure.txt"
]

# Exécution
print(f"Début du traitement de {len(input_files)} fichiers")
total_start = time.time()

for in_file, out_file in zip(input_files, output_files):
    process_file(in_file, out_file)

print(f"\nTemps total: {(time.time()-total_start)/60:.1f} minutes")

Début du traitement de 5 fichiers

Traitement de weighted_BIOGRID_humain.txt


Lecture fichier: 86981it [00:00, 204808.61it/s]


Calcul des poids des nœuds...


Détection des complexes:   0%|          | 0/11120 [00:00<?, ?it/s]

KeyboardInterrupt: 

ClusterONE

In [3]:
import networkx as nx
from collections import defaultdict
import numpy as np

def read_weighted_interactions(input_file):
    """Lit un fichier d'interactions pondérées avec vérification des chemins"""
    if not os.path.exists(input_file):
        raise FileNotFoundError(f"Fichier introuvable : {input_file}")
    
    G = nx.Graph()
    with open(input_file, 'r') as f:
        header = next(f)
        if not header.startswith("protein1"):
            f.seek(0)  # Retour au début si pas d'en-tête
            
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    protein1, protein2, weight = parts[0], parts[1], float(parts[2])
                    G.add_edge(protein1, protein2, weight=weight)
                except ValueError:
                    continue
    return G

def calculate_cohesiveness(graph, cluster, node, penalty=2):
    """Version améliorée du calcul de cohésion avec normalisation"""
    internal = sum(graph[node][neighbor]['weight'] 
               for neighbor in cluster 
               if neighbor in graph[node])
    
    external = sum(graph[node][neighbor]['weight']
               for neighbor in graph.neighbors(node)
               if neighbor not in cluster)
    
    # Normalisation par la taille du cluster
    norm_factor = max(1, len(cluster)**0.5)
    return (internal - penalty * external) / norm_factor

def clusterone(input_file, output_file, 
               density_threshold='auto', 
               penalty=2, 
               min_size=3, 
               overlap_threshold=0.8):
    """
    Version optimisée de ClusterONE avec :
    - Détection automatique de la densité seuil
    - Paramètres par défaut des auteurs
    - Gestion robuste des erreurs
    """
    
    # Lecture du graphe avec vérification
    try:
        graph = read_weighted_interactions(input_file)
    except Exception as e:
        print(f"Erreur lecture {input_file} : {str(e)}")
        return
    
    # Détection auto de la densité seuil (si 'auto')
    if density_threshold == 'auto':
        densities = [nx.density(graph.subgraph([n])) for n in graph.nodes()]
        density_threshold = max(0.1, min(0.3, np.percentile(densities, 75)))
    
    # Algorithme ClusterONE
    clusters = []
    nodes = list(graph.nodes())
    
    # Étape 1: Croissance gourmande
    for seed in nodes:
        if any(seed in cluster for cluster in clusters):
            continue
            
        cluster = {seed}
        neighbors = set(graph.neighbors(seed))
        
        while neighbors:
            best_node = max(
                neighbors,
                key=lambda n: calculate_cohesiveness(graph, cluster, n, penalty),
                default=None
            )
            
            if best_node is None:
                break
                
            new_cluster = cluster | {best_node}
            subgraph = graph.subgraph(new_cluster)
            actual_edges = sum(d['weight'] for _, _, d in subgraph.edges(data=True))
            possible_edges = len(new_cluster) * (len(new_cluster) - 1) / 2
            density = actual_edges / possible_edges if possible_edges > 0 else 0
            
            if density >= density_threshold:
                cluster.add(best_node)
                neighbors.remove(best_node)
                neighbors.update(
                    n for n in graph.neighbors(best_node)
                    if n not in cluster
                )
            else:
                break
        
        if len(cluster) >= min_size:
            clusters.append(cluster)
    
    # Étape 2: Fusion des clusters (seuil 0.8 comme recommandé)
    merged_clusters = []
    used = set()
    
    for i in range(len(clusters)):
        if i in used:
            continue
        current = clusters[i]
        for j in range(i+1, len(clusters)):
            if j in used:
                continue
            overlap = len(current & clusters[j]) / min(len(current), len(clusters[j]))
            if overlap > overlap_threshold:
                current |= clusters[j]
                used.add(j)
        merged_clusters.append(current)
    
    # Écriture des résultats avec vérification du dossier
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w') as f:
        for i, cluster in enumerate(
            sorted(merged_clusters, key=lambda x: -len(x)), 1
        ):
            f.write(f"{i}\t{';'.join(sorted(cluster))}\n")

# Paramètres des auteurs
DEFAULT_PARAMS = {
    'density_threshold': 'auto',  # Détection automatique
    'penalty': 2,
    'min_size': 3,
    'overlap_threshold': 0.8
}


# Exemple d'utilisation pour plusieurs fichiers
input_files = ["/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt"
    ]
output_files = ["/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/ClusterONE_complexes_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/ClusterONE_complexes_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/ClusterONE_complexes_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/ClusterONE_complexes_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/ClusterONE_complexes_STRING_levure.txt"
    ]

for in_file, out_file in zip(input_files, output_files):
    print(f"Traitement de {in_file}...")
    clusterone(in_file, out_file, **DEFAULT_PARAMS)     # Complexes d'au moins 3 protéines

Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt...


Détection des complexes:   0%|          | 0/11120 [56:04<?, ?it/s]

Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt...


Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_DIP_levure.txt...
Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt...
Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt...


COACH

In [1]:
import networkx as nx
import itertools
from collections import defaultdict
import os
from tqdm import tqdm

def read_weighted_ppi(file_path):
    """Lecture du réseau PPI pondéré avec barre de progression"""
    G = nx.Graph()
    # D'abord compter le nombre de lignes pour la barre de progression
    with open(file_path, 'r') as f:
        total_lines = sum(1 for _ in f)
    
    # Maintenant lire le fichier avec barre de progression
    with open(file_path, 'r') as f:
        next(f)  # Skip header
        for line in tqdm(f, total=total_lines-1, desc="Chargement du réseau"):
            parts = line.strip().split()
            if len(parts) >= 3:
                try:
                    protein1, protein2, weight = parts[0], parts[1], float(parts[2])
                    G.add_edge(protein1, protein2, weight=weight)
                except ValueError:
                    continue
    return G

def calculate_cluster_cohesion(graph, nodes):
    """Calcul de cohésion avec gestion des cas particuliers"""
    if len(nodes) < 2:
        return 0
    
    subgraph = graph.subgraph(nodes)
    total_weight = sum(data['weight'] for _, _, data in subgraph.edges(data=True))
    possible_edges = len(nodes) * (len(nodes) - 1) / 2
    
    if possible_edges == 0:
        return 0
    return total_weight / possible_edges

def detect_potential_cores(graph, min_degree=2, min_weight=0.2):
    """Détection des coeurs avec barre de progression"""
    cores = set()
    nodes = list(graph.nodes())
    avg_degree = sum(dict(graph.degree(weight='weight')).values()) / len(nodes)
    
    for node in tqdm(nodes, desc="Détection des coeurs"):
        neighbors = list(graph.neighbors(node))
        if len(neighbors) < max(min_degree, avg_degree * 0.5):
            continue
            
        # Compter les connexions fortes
        strong_links = sum(1 for neighbor in neighbors 
                         if graph[node][neighbor]['weight'] >= min_weight)
        
        if strong_links >= 2:
            cores.add(node)
    
    return cores

def form_core_complexes(graph, cores, min_similarity=0.4):
    """Formation des complexes de coeur avec progression"""
    core_graph = nx.Graph()
    core_list = list(cores)
    
    # Utilisation de tqdm pour les combinaisons
    for u, v in tqdm(itertools.combinations(core_list, 2), 
                    total=len(core_list)*(len(core_list)-1)//2,
                    desc="Formation des coeurs"):
        if graph.has_edge(u, v):
            weight = graph[u][v]['weight']
            if weight >= min_similarity:
                core_graph.add_edge(u, v, weight=weight)
    
    return [set(c) for c in nx.connected_components(core_graph)]

def grow_complexes(graph, core_complexes, min_cohesion=0.3):
    """Croissance des complexes avec barre de progression"""
    complexes = []
    
    for core in tqdm(core_complexes, desc="Croissance des complexes"):
        complex_nodes = set(core)
        border = set(n for node in core for n in graph.neighbors(node)) - complex_nodes
        progress_bar = tqdm(total=len(border), leave=False, desc="Attachements")
        
        while border:
            best_candidate = None
            best_gain = 0
            
            for candidate in border:
                temp_complex = complex_nodes | {candidate}
                new_cohesion = calculate_cluster_cohesion(graph, temp_complex)
                
                if new_cohesion >= min_cohesion:
                    gain = sum(graph[candidate][n]['weight'] for n in complex_nodes 
                              if graph.has_edge(candidate, n))
                    
                    if gain > best_gain:
                        best_gain = gain
                        best_candidate = candidate
            
            if best_candidate:
                complex_nodes.add(best_candidate)
                border.remove(best_candidate)
                border.update(n for n in graph.neighbors(best_candidate) 
                           if n not in complex_nodes)
                progress_bar.update(1)
            else:
                break
        
        progress_bar.close()
        if len(complex_nodes) >= 3:
            complexes.append(complex_nodes)
    
    return complexes

def coach(input_file, output_file, min_degree=1, min_weight=0.1, min_similarity=0.2, min_cohesion=0.15, min_size=2):
    """Version corrigée avec paramètres ajustables"""
    print(f"\n{'='*50}\nTraitement de {input_file}\n{'='*50}")
    
    # 1. Chargement du réseau
    graph = read_weighted_ppi(input_file)
    if not graph.edges():
        print("Réseau vide - vérifiez le fichier d'entrée")
        return
    
    # 2-4. Détection et croissance des complexes (avec nouveaux paramètres)
    cores = detect_potential_cores(graph, min_degree, min_weight)
    core_complexes = form_core_complexes(graph, cores, min_similarity)
    complexes = grow_complexes(graph, core_complexes, min_cohesion)
    
    # 5. Sauvegarde avec vérification
    final_complexes = [c for c in complexes if len(c) >= min_size]
    
    print(f"\n{'='*50}")
    print(f"Résultats pour {input_file}:")
    print(f"- Coeurs initiaux: {len(cores)}")
    print(f"- Complexes après croissance: {len(complexes)}")
    print(f"- Complexes finals (taille >= {min_size}): {len(final_complexes)}")
    
    if not final_complexes:
        print("Aucun complexe à sauvegarder - vérifiez les paramètres")
        return
    
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w') as f:
        for i, comp in enumerate(tqdm(sorted(final_complexes, key=lambda x: -len(x)), desc="Sauvegarde"), 1):
            f.write(f"{i}\t{';'.join(sorted(comp))}\n")
    
    print(f"Fichier sauvegardé: {output_file}")
    print(f"{'='*50}\n")

# Paramètres d'entrée/sortie
input_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt"
]

output_files = [
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_BIOGRID_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_BIOGRID_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_DIP_levure.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_humain.txt",
    "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_levure.txt"
]

# Exécution avec gestion des erreurs
for in_file, out_file in zip(input_files, output_files):
    try:
        coach(in_file, out_file)
    except Exception as e:
        print(f"\n{'!'*50}\nErreur lors du traitement de {in_file}: {str(e)}\n{'!'*50}")


Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt


Croissance des complexes: 100%|██████████| 8/8 [25:42<00:00, 192.81s/it]   



Résultats pour /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_humain.txt:
- Coeurs initiaux: 7683
- Complexes après croissance: 7
- Complexes finals (taille >= 2): 7


Sauvegarde: 100%|██████████| 7/7 [00:00<00:00, 1771.78it/s]


Fichier sauvegardé: /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_BIOGRID_humain.txt


Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_BIOGRID_levure.txt


...

Fichier sauvegardé: /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_humain.txt


Traitement de /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt


Croissance des complexes: 100%|██████████| 13/13 [02:48<00:00, 12.93s/it]



Résultats pour /Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/weighted_networks/weighted_STRING_levure.txt:
- Coeurs initiaux: 3348
- Complexes après croissance: 13
- Complexes finals (taille >= 2): 13


Sauvegarde: 100%|██████████| 13/13 [00:00<00:00, 7453.99it/s]

Fichier sauvegardé: /Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_levure.txt

